# Reproduce Directional Analysis 1

This notebook provides the interactive SCIViewer workflow for Directional Analysis 1.

Use it in one of two ways:

1. **Interactive mode:** load the AnnData object, open SCIViewer, recreate the directional selection, export the selected cells and projection-correlation table, then regenerate Table S4/Table S5 outputs.
2. **Saved-export mode:** use the saved SCIViewer exports from the original analysis to regenerate the submitted supplemental tables.

Large `.h5ad` files and generated outputs are distributed through GEO accession `GSE348416` or generated locally by this workflow.


In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE

# Set HCMV_DATA_DIR to the folder containing downloaded data/intermediate files.
DATA_DIR = Path(os.environ.get("HCMV_DATA_DIR", REPO_ROOT / "data")).expanduser().resolve()
SUPPLEMENT_DIR = Path(os.environ.get("HCMV_SUPPLEMENT_DIR", REPO_ROOT / "supplemental")).expanduser().resolve()

OUTPUT_DIR = REPO_ROOT / "outputs" / "directional_analysis_1"
TABLE_OUTPUT_DIR = REPO_ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANN_DATA = DATA_DIR / "infected_cells_3dpi_cmv_data.h5ad"

# Saved intermediate files from the original interactive SCIViewer analysis.
SAVED_SELECTED_CELLS = DATA_DIR / "selected_cells_04Nov24_dir1_dpi3_infected.only.host.only.csv"
SAVED_PROJ_CORR_XLSX = DATA_DIR / "results_04Nov24_dir1_dpi3_infected.only.host.only.proj_correlation.xlsx"
SAVED_PROJ_CORR_CSV = DATA_DIR / "results_04Nov24_dir1_dpi3_infected.only.host.only.proj_correlation.csv"
SAVED_GSEA_WORKBOOK = DATA_DIR / "04Nov24_dir1_dpi3_infected.only.host.only.filtgenes.fdr_10_to_genes_with_regulation_mapping.xlsx"

# Outputs from a newly recreated interactive selection.
NEW_SELECTED_CELLS = OUTPUT_DIR / "selected_cells_directional_analysis_1_recreated.csv"
NEW_PROJ_CORR_CSV = OUTPUT_DIR / "projection_correlation_directional_analysis_1_recreated.csv"

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR:", DATA_DIR)
print("AnnData exists:", ANN_DATA.exists())
print("Saved selected-cell file exists:", SAVED_SELECTED_CELLS.exists())
print("Saved projection-correlation export exists:", SAVED_PROJ_CORR_XLSX.exists())
print("Saved DA1 GSEA workbook exists:", SAVED_GSEA_WORKBOOK.exists())


## Load the 3 dpi infected-cell AnnData object

This AnnData object is distributed through GEO accession `GSE348416`.


In [ ]:
import pandas as pd

try:
    import scanpy as sc
except ImportError as exc:
    raise ImportError("Install scanpy/anndata in this Jupyter environment before running the interactive workflow.") from exc

adata = sc.read_h5ad(ANN_DATA)
adata


## Open SCIViewer and recreate the direction

Run the next cell to open SCIViewer. Recreate the Directional Analysis 1 selection interactively. When you are satisfied with the direction/selected cells, continue to the export cell below.

The saved selection used for the manuscript is `selected_cells_04Nov24_dir1_dpi3_infected.only.host.only.csv`. You can compare against it after exporting.


In [ ]:
import os
import sys

# SCIViewer uses py5, which needs Java 17. The conda environment installs openjdk.
if 'JAVA_HOME' not in os.environ:
    conda_java_home = Path(os.environ.get('CONDA_PREFIX', '')) / 'lib' / 'jvm'
    if conda_java_home.exists():
        os.environ['JAVA_HOME'] = str(conda_java_home)

ip = get_ipython()
if sys.platform == 'darwin':
    ip.run_line_magic('gui', 'osx')
ip.run_line_magic('load_ext', 'py5')

try:
    from sciviewer import SCIViewer
except ImportError as exc:
    raise ImportError(
        "SCIViewer is not installed in this Jupyter kernel. "
        "Select the `Python (HCMV sciviewer)` kernel, restart, and run from the top."
    ) from exc

svobj_dir1 = SCIViewer(adata, embedding_name="X_umap", use_raw=False)
svobj_dir1.explore_data()


## Export the recreated SCIViewer selection

Run this after you have recreated the directional selection in SCIViewer. It writes generated files under `outputs/directional_analysis_1/`.


In [ ]:
selected = pd.DataFrame(svobj_dir1.selected_cells)
selected.to_csv(NEW_SELECTED_CELLS, index=False)

proj_corr = svobj_dir1.results_proj_correlation.dropna(subset=["P"]).copy()
proj_corr.to_csv(NEW_PROJ_CORR_CSV, index=True)

print("Wrote:", NEW_SELECTED_CELLS)
print("Wrote:", NEW_PROJ_CORR_CSV)
print("Selected cells:", selected.shape)
print("Projection-correlation rows:", proj_corr.shape)
proj_corr.sort_values("P").head()


## Choose source for downstream reproduction

Use `USE_SAVED_EXPORT = True` to regenerate the submitted tables from the saved intermediate file. Set it to `False` after exporting a newly recreated SCIViewer selection above.


In [ ]:
USE_SAVED_EXPORT = True

projection_correlation_source = SAVED_PROJ_CORR_XLSX if USE_SAVED_EXPORT else NEW_PROJ_CORR_CSV
print("Using projection-correlation source:", projection_correlation_source)
print("Exists:", projection_correlation_source.exists())


## Regenerate Table S4 and Table S5

This imports the cleaned deterministic helper functions from `scripts/directional_analysis/reproduce_directional_analysis_1.py`. Table S4 and Table S5 are the manuscript supplemental outputs for Directional Analysis 1.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "scripts" / "directional_analysis"))
from reproduce_directional_analysis_1 import write_table_s4, write_table_s5

TABLE_S4_OUT = TABLE_OUTPUT_DIR / "Table_S4_supplement_directional_analysis_1_raw_results.xlsx"
TABLE_S5_OUT = TABLE_OUTPUT_DIR / "Table_S5_supplement_GSEA_directional_analysis_1.xlsx"

write_table_s4(projection_correlation_source, TABLE_S4_OUT)
write_table_s5(SAVED_GSEA_WORKBOOK, TABLE_S5_OUT)

print("Wrote:", TABLE_S4_OUT)
print("Wrote:", TABLE_S5_OUT)


## Optional comparison with submitted supplemental tables

This optional cell compares regenerated tables with reference copies of the submitted supplemental tables, when those files are available.


In [ ]:
expected_s4 = SUPPLEMENT_DIR / "Table_S4_supplement_directional_analysis_1_raw_results.xlsx"
expected_s5 = SUPPLEMENT_DIR / "Table_S5_supplement_GSEA_directional_analysis_1.xlsx"

def compare_excel_values(expected_path, actual_path):
    if not expected_path.exists():
        print("Missing expected file, skipping:", expected_path)
        return
    expected = pd.read_excel(expected_path).fillna("").astype(str)
    actual = pd.read_excel(actual_path).fillna("").astype(str)
    common = [col for col in expected.columns if col in actual.columns]
    print(expected_path.name)
    print("  expected shape:", expected.shape)
    print("  actual shape:  ", actual.shape)
    print("  columns equal: ", list(expected.columns) == list(actual.columns))
    print("  values equal:  ", expected[common].equals(actual[common]))

compare_excel_values(expected_s4, TABLE_S4_OUT)
compare_excel_values(expected_s5, TABLE_S5_OUT)


## Plot Directional Analysis 1 GSEA panel

This creates the manuscript-style GSEA NES bar plot from the regenerated Table S5 workbook. The default FDR cutoff is `0.1`, matching the manuscript caption for Directional Analysis 1.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "scripts" / "figures"))
from plot_directional_gsea_panel import load_gsea_table, select_pathways, plot_gsea_bar, write_selected_pathways

FIGURE_OUTPUT_DIR = REPO_ROOT / "outputs" / "figures" / "directional_analysis_1"
DA1_GSEA_PANEL_SVG = FIGURE_OUTPUT_DIR / "directional_analysis_1_gsea_fdr_0.1.svg"
DA1_GSEA_PANEL_PNG = FIGURE_OUTPUT_DIR / "directional_analysis_1_gsea_fdr_0.1.png"
DA1_GSEA_PANEL_TSV = FIGURE_OUTPUT_DIR / "directional_analysis_1_gsea_fdr_0.1_plotted_pathways.tsv"

gsea_table = load_gsea_table(TABLE_S5_OUT)
plotted_pathways = select_pathways(gsea_table, fdr_cutoff=0.1, top_n=None)
plot_gsea_bar(plotted_pathways, DA1_GSEA_PANEL_SVG, title="Directional Analysis 1 GSEA")
plot_gsea_bar(plotted_pathways, DA1_GSEA_PANEL_PNG, title="Directional Analysis 1 GSEA")
write_selected_pathways(plotted_pathways, DA1_GSEA_PANEL_TSV)

print("Wrote:", DA1_GSEA_PANEL_SVG)
print("Wrote:", DA1_GSEA_PANEL_PNG)
print("Wrote:", DA1_GSEA_PANEL_TSV)
print("Plotted pathways:", plotted_pathways.shape[0])
display(plotted_pathways[["Term", "NES", "FDR q-val"]])


## Plot Directional Analysis 1 top-gene dotplots

These reproduce the Figure 5C/D-style dotplots for the top positively and negatively correlated genes from Directional Analysis 1. The SCIViewer selection and Table S4 use the 3 dpi infected-cell object above, but the dotplots in the manuscript are drawn across all 3 dpi infection-state groups from the all-cell AnnData object (`cmv.srt.soupx.filt.h5ad`).

The grouping is: Mock, Bystander, Marginal Infection, then High Infection cells binned by percent HCMV transcripts. Dot color is z-scored average expression per gene, clipped to -2 to 2; dot size is percent of cells expressing the gene. For the dotplot only, expression is read from `adata.raw` and summarized with Seurat DotPlot-style averaging to mirror the original R/scCustomize plotting convention.


In [ ]:
import importlib

import anndata as ad
import plot_directional_gene_dotplots as dotplots

# Reload the plotting module when rerunning this cell in an active Jupyter kernel.
dotplots = importlib.reload(dotplots)

DOTPLOT_ADATA = DATA_DIR / "cmv.srt.soupx.filt.h5ad"

dotplot_adata = ad.read_h5ad(DOTPLOT_ADATA)
dotplot_adata.obs["figure5_dotplot_group"] = dotplots.make_dotplot_group(dotplot_adata.obs, dpi="3dpi")
dotplot_adata = dotplot_adata[~pd.isna(dotplot_adata.obs["figure5_dotplot_group"]), :].copy()
dotplot_correlations = dotplots.load_correlation_table(TABLE_S4_OUT)
dotplot_results = []

for direction, top_n, title in [
    ("positive", 33, "sciViewer top positively correlated\ngene expression in dpi3 highly infected cells"),
    ("negative", 30, "sciViewer top negatively correlated\ngene expression in dpi3 highly infected cells"),
]:
    result = dotplots.build_dotplot(
        dotplot_adata,
        dotplot_correlations,
        direction=direction,
        output_dir=FIGURE_OUTPUT_DIR,
        top_n=top_n,
        use_raw=True,
        file_prefix="directional_analysis_1",
        title=title,
        figure_order=True,
        average_method="seurat",
    )
    dotplot_results.append(result)
    print("Wrote:", result.figure_path)
    print("Wrote:", result.table_path)

[(r.direction, r.figure_path.name, r.table_path.name, len(r.genes)) for r in dotplot_results]


## Optional: compare a recreated selected-cell export to the original saved selection

Run this only after exporting a recreated selection. Exact equality is not required for a new interactive selection, but this comparison is useful for checking whether the same direction was recovered.


In [ ]:
if NEW_SELECTED_CELLS.exists() and SAVED_SELECTED_CELLS.exists():
    old = pd.read_csv(SAVED_SELECTED_CELLS).fillna("")
    new = pd.read_csv(NEW_SELECTED_CELLS).fillna("")
    print("Original selected cells shape:", old.shape)
    print("Recreated selected cells shape:", new.shape)
    common_cols = [col for col in old.columns if col in new.columns]
    if common_cols and old.shape == new.shape:
        print("Shared-column values equal:", old[common_cols].astype(str).equals(new[common_cols].astype(str)))
    display(old.head())
    display(new.head())
else:
    print("Selection comparison skipped. Export a recreated selection first, and confirm the saved selected-cell CSV exists.")
